In [51]:
#! git init  # Only if the repo isn't initialized
# ! git remote add origin https://github.com/hs-radio/SEEK_jobs  # Only if not added

# git add *.ipynb  # Add all Jupyter Notebooks
# git commit -m "Added Jupyter Notebook files"
# git push origin main
! git branch

In [1]:
import requests
import importlib
from bs4 import BeautifulSoup
import pandas as pd
import snowflake.connector
import sqlalchemy as db
import numpy as np
import import_ipynb
import re
from datetime import datetime, timedelta
import snowflake_functions as sf  # Import the notebook as a module
import transform_data as td 
import extract_data as ed 
from skills_array import get_skills_array
skills_array = get_skills_array() # get full list of skills

In [36]:
# upload latest function
importlib.reload(sf)
importlib.reload(ed)
importlib.reload(td)

# connection snowflake
engine = sf.connect_snowflake()
connection = engine.connect()

# intialise tables that will later be loaded to snowflake
jt_columns = ['TITLE', 'COMPANY', 'LOCATION', 'EMPLOYMENT_TYPE', 'SALARY', 'PAY_PERIOD', 'POST_DATE']
jst_columns = ['JOB_ID', 'SKILL_ID']
job_table = pd.DataFrame(columns= jt_columns)
job_skills_table = pd.DataFrame(columns=jst_columns)

# latest job_id in snowflake server
last_job_id = sf.last_job_id(connection) # change to get all job_ids and avoid used ones.

# go through many pages
max_pages = 1
for j in range(max_pages):
    
    # get all job URL links
    # full_urls = ed.get_URLs_to_jobs(j) 
    
    # go through all the URLs and extract relevant data.
    for url in full_urls:
        
        # Scrape job add URL
        response = requests.get(url) 
        soup = BeautifulSoup(response.content, "html.parser") 
        
        # Enter data into jobs table
        job_props, req_skills = ed.get_job_data(soup, skills_array)
        parsed_salary = td.parse_salary(job_props[4]) # clean up salary
        job_table = td.update_job_table(job_props, parsed_salary, job_table)
    
        # Enter data into job_skills table.
        df_skill_ids = sf.get_skill_ids(req_skills, connection)
        job_skills_table = td.update_job_skills_table(job_props, parsed_salary, job_skills_table, df_skill_ids, last_job_id + 1 + j)

# Append the tables in the snowflake database
job_table_entry.to_sql('jobs', con=engine, if_exists='append', index=False)
job_skills_table_entry.to_sql('job_skills', con=engine, if_exists='append', index=False)


# Close connection
connection.close()

Connected to Snowflake!


KeyboardInterrupt: 

In [38]:
# Close connection
connection.close()

In [44]:
job_table
# job_skills_table
# job_id

,TITLE,COMPANY,LOCATION,EMPLOYMENT_TYPE,SALARY,PAY_PERIOD,POST_DATE
0,Data Engineer | Mid-Level,Billigence,Sydney NSW,Contract/Temp,None,None,2025-02-25
1,Principal Data Engineer,Attribute Group,Sydney NSW,Contract/Temp,None,None,2025-02-18
2,Senior Data Engineer & Analytics Specialist,OurPath,Sydney NSW,Full time,120000,annually,2025-02-16
3,GCP Data Engineer,SALT SEARCH PTY LTD,Sydney NSW,Full time,150000,annually,2025-02-24
4,Data Engineer - Azure,Aurec,"Parramatta, Sydney NSW",Full time,167000,annually,None
5,Senior Data Engineer (Contexa),FinXL IT Professional Services,Sydney NSW,Contract/Temp,None,None,None
